# పాఠం 04 - టూల్ ఉపయోగం డిజైన్ నమూనా

ఈ పాఠంలో మీరు Microsoft Agent Framework (Python) ఉపయోగించి AI ఏజెంట్ల కోసం **టూల్ ఉపయోగం** డిజైన్ నమూనాను నేర్చుకుంటారు. మేము కవర్ చేస్తాము:

- `@tool` డెకరేటర్ తో మరియు టైప్ చేసిన పారామీటర్లతో ఫంక్షన్ టూల్స్ నిర్వచించడం
- മോడൽ ప్రతీ టూల్ ఏమి చేస్తుంది తెలుసుకునేందుకు టూల్ స్కీమాలు అందించడం
- `approval_mode` తో టూల్ కార్యాచరణని నియంత్రించడం
- Pydantic మోడల్స్ మరియు `response_format` ద్వారా **నిర్మిత అవుట్పుట్** ఇవ్వడం

సన్నివేశం ఒక **ప్రయాణ బుకింగ్ ఏజెంట్** ఇది గమ్యస్థానాలను చూడగలదు, అందుబాటును తనిఖీ చేయగలదు, మరియు ఫ్లైట్ సమాచారాన్ని పొందగలదు.


## సెటప్


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool డెకరేటర్‌తో టూల్స్ నిర్వచించడం

`@tool` డెకరేటర్ ఒక సాధారణ Python ఫంక్షన్‌ని ఏజెంట్ కాల్ చేయగల టూల్‌గా మార్చుతుంది.
ప్రధాన అంశాలు:

- **డాక్స్ట్రింగ్** మోడల్ చూసే టూల్ వివరణ అవుతుంది.
- **టైప్ అనోటేషన్లు** (`Annotated` సహా వివరణలతో) టూల్ స్కీమాను నిర్వచిస్తాయి.
- `approval_mode` యూజర్ ప్రతి కాల్‌ను అమలు avant-garde ముందు అంగీకరించాలా కాదా నియంత్రిస్తుంది.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## బహుళ పరికరాలతో ఏజెంట్‌ని సృష్టించడం

మూడు పరికరాలను కూడా క్లయింట్‌కు అందించండి, తద్వారా మోడల్ అవసరమైన వాటిని పిలిచి వినియోగదారుడి ప్రశ్నకు సమాధానం ఇవ్వగలదు.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## టూల్స్‌తో స్ట్రక్చర్డ్ అవుట్పుట్

`response_format` ను Pydantic మోడల్‌గా సెట్ చేసేలా చేస్తే, ఏజెంట్ స్వేచ్ఛాత్మక పాఠ్యానికిమాత్రం కాకుండా బాగా టైప్ చేయబడిన JSON ఆబ్జెక్ట్‌ను తిరిగి ఇవ్వవలసి ఉంటుంది. ఇది దిగువ కోడ్ ఫలితాన్ని ప్రోగ్రామాటిక్ గా తీసుకోవాల్సిన సందర్భాల్లో ఉపయోగపడుతుంది.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Tool Approval Patterns

`@tool` పై `approval_mode` పరామితి టూల్ పిలుపులు అమలులోకి రావడానికి మానవ అనుమతి అవసరమా లేకపోవడం నియంత్రిస్తుంది:

| మోడ్ | ప్రవర్తన |
|---|---|
| `"never_require"` | టూల్ ఆటోమేటిక్ గా అమలవుతుంది — వినియోగదారు నిర్ధారణ అవసరం లేదు. |
| `"always_require"` | ప్రతి పిలుపు అమలవ్వడానికి ముందు వినియోగదారు ద్వారా ఆమోదించబడాలి. |

పక్కవ ప్రభావాలు ఉన్న టూల్స్ కోసం (ఉదా: విమానం బుక్ చేయడం, క్రెడిట్ కార్డ్ ఛార్జ్ చేయడం) `"always_require"` ను పాడయండి, తద్వారా మానవుడి నియంత్రణలో ఉంటారు.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## సారాంశం

ఈ పాఠంలో మీరు ఎలా చేయాలో నేర్చుకున్నారు:

1. టైప్ చేయబడిన ప్యారామీటర్లు మరియు టూల్ స్కీమాగా పనిచేసే డాక్స్ట్రింగ్‌లతో `@tool` డెకరేటర్ ఉపయోగించి **టూల్స్‌ను నిర్వచించండి**.
2. ఏజెంట్ క్వెరీలను సమాధానం చెప్పడానికి ఈ టూల్స్‌ను వరుసగా పిలవడం కోసం **బహుళ టూల్స్‌ను సంయోగించండి**.
3. `response_format`గా పిడాంటిక్ మోడను పంపించి **సంరచనాత్మక అవుట్పుట్‌ను తిరిగి ఇవ్వండి**.
4. సున్నితమైన ఆపరేషన్ల కోసం మానవులను చక్రంలో ఉంచడానికి `approval_mode`తో **టూల్ ఆమోదాన్ని నియంత్రించండి**.

ఈ నమూనాలు భద్రతగా బయటి సిస్టమ్‌లతో ఇంటరాక్ట్ చేయగల నమ్మదగిన, ఉత్పాదకత-సిద్ధ ఏజెంట్‌లు తయారు చేయడానికి వ్యవస్థాపకాన్ని ఏర్పాటు చేస్తాయి.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**నివారణ**:  
ఈ డాక్యూమెంట్ AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ద్వారా అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటిక్ అనువాదాల్లో పొరపాట్లు లేదా లోపాలు ఉండవచ్చు. స్థానీయ భాషలో ఉన్న అసలు డాక్యూమెంట్‌ను అధికారిక మూలం గా పరిగణించాలి. ముఖ్యమైన సమాచారానికి, నిపుణుల మానవ అనువాదం సలహాదాయకం. ఈ అనువాదం వాడకంలో జరిగే ఏ అవగాహనా లోపాలు లేదా తప్పుదోవలకు మేము బాధ్యులు కారు.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
